# Governance Dashboard — QuickSight Analysis for Drift Monitoring

> 📊 **Best viewed on [nbviewer](https://nbviewer.org/github/aws-samples/sample-mlops-bestpractices/blob/main/sagemaker-automated-drift-and-trend-monitoring/notebooks/4_governance_dashboard.ipynb)** — GitHub's renderer strips JavaScript that powers interactive output cells (Evidently reports, plotly charts, ipywidgets). nbviewer (run by Project Jupyter) renders them in full.


This notebook programmatically creates a complete QuickSight dashboard — no manual UI steps.

**Schema awareness:** the feature-drift dataset (Sheet 3) and the inference dataset both expose `monitoring_run_id`. The drift Lambda back-fills this column on `inference_responses` for every row it scored, so the cross-dataset JOIN is now an exact foreign key (was a 24-hour time-window approximation in earlier versions).

**Visuals — Sheet 1 (Model Drift Trends):**
1. ROC-AUC: Baseline vs Current Over Time
2. Model Performance Metrics Over Time
3. ROC-AUC Trend by Model Version
4. Model-Drift Verdict Rate by Model Package ARN
5. Model-Drift Verdict Rate by Endpoint
6. Performance by Training Snapshot
7. Model Lineage Audit (per monitoring run)
8. Latest Current ROC-AUC (KPI)
9. Confusion Matrix Over Time (TP/FP/TN/FN)
10. ROC-AUC Degradation % Over Time
11. Ground-Truth Coverage Over Time (labels / predictions)

**Visuals — Sheet 2 (Data Drift Trends):**
1. Data Drift Share Over Time
2. Drifted Features Count Over Time
3. Drift Alerts Timeline
4. Data Drift Share by Model Version
5. Data Drift Share by Endpoint
6. Inference Volume vs Drift Share Correlation
7. Latest Data Drift Share (KPI)
8. Source Data — monitoring_responses (table)
9. Prediction Score Distribution Over Time (leading indicator)
10. Drift Verdict Sample Size Per Run

**Visuals — Sheet 3 (Feature Drift Trends):**
1. Feature Drift Score Timeline (per-feature line chart)
2. Top 15 Most-Drifting Features (All Time)
3. Drift Severity Distribution by Feature (Top 15)
4. Feature Drift Heatmap (Features × Time)
5. Feature Drift Details (per run × feature table)
6. Highest Current Drift Score (KPI)
7. Feature Drift Heatmap (Features × Model Version)
8. Max Feature Drift Score Per Run (worst-feature signal)
9. Repeat-Offender Features (# of runs where drift_detected=TRUE)
10. Raw drift_score — p-value tests (KS / Chi-square) — LOWER = more drift
11. Raw drift_score — distance tests (Wasserstein / Jensen-Shannon / PSI) — HIGHER = more drift

## What this notebook does

This notebook builds (or refreshes) the QuickSight **governance dashboard** — the
inference, drift, feature-drift, and prediction-accuracy visuals — on top of the
Athena tables written by the drift-monitoring pipeline.

**All of the logic lives in one place:** `src/governance/create_governance_dashboard.py`.
That module is the single source of truth for every dataset, Athena view, visual, analysis,
and dashboard definition. It is also what `main.py dashboard create` calls. This notebook is
a thin driver over that module — it does **not** redefine any datasets or visuals inline, so
the notebook and the CLI can never drift out of sync.

> Because everything is defined in the module, the feature-drift visuals automatically use the
> test-agnostic **`drift_magnitude`** field (higher = more drift, regardless of which statistical
> test Evidently picked) instead of the ambiguous raw `drift_score`.

## 1. Setup & Configuration

In [13]:
import sys, boto3
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / '.env')

from src.config.config import AWS_DEFAULT_REGION, QUICKSIGHT_IDENTITY_REGION, ATHENA_DATABASE

# Everything the notebook does is delegated to this module — the single
# source of truth for datasets, the feature_drift_detail view, visuals,
# the analysis, and the dashboard. No dataset/visual is redefined here.
from src.governance import create_governance_dashboard as gov

# Account ID for the QuickSight/Athena calls below.
sts = boto3.client('sts', region_name=AWS_DEFAULT_REGION)
ACCOUNT_ID = sts.get_caller_identity()['Account']

print(f'Account:                    {ACCOUNT_ID}')
print(f'Data-plane region (Athena): {AWS_DEFAULT_REGION}')
print(f'QuickSight identity region: {QUICKSIGHT_IDENTITY_REGION}')
print(f'Database:                   {ATHENA_DATABASE}')

Account:                    329430715989
Data-plane region (Athena): us-east-1
QuickSight identity region: us-east-1
Database:                   fraud_detection


> ⚠️ **First-time QuickSight setup required (once per AWS account).**
>
> If this is a brand-new account, `describe_account_settings` in the next cell will fail with `ResourceNotFoundException` or similar. QuickSight has to be signed up for in the AWS console first — the CLI/notebook can't do it for you. **5-minute one-time setup:**
>
> 1. AWS console → search **QuickSight** → **Sign up for QuickSight**.
> 2. Edition: **Enterprise** (Standard doesn't support the Definition API this notebook uses).
> 3. Auth: **Use IAM federated identities and QuickSight-managed users** (default).
> 4. Region: pick your **identity region** — QuickSight fixes this per account at sign-up. Usually `us-east-1`. If yours is different, set `QUICKSIGHT_IDENTITY_REGION=<region>` in `.env` before running this notebook.
> 5. QuickSight console → username (top right) → **Manage QuickSight** → **Manage users** → **Invite users** — add your IAM/SSO identity as **Author** or **Admin**.
> 6. Same page → **Security & permissions** → **Manage QuickSight access to AWS services** → check **Amazon S3** (select the base stack's data bucket) and **Amazon Athena**. Save.
>
> Verify by visiting [https://quicksight.aws.amazon.com/](https://quicksight.aws.amazon.com/) — you should see the QuickSight home page. Then rerun the cell below.
>
> Full prerequisites + troubleshooting: see the README's **QuickSight prerequisites (one-time per account)** section.

## 2. Verify QuickSight Subscription

In [14]:
from botocore.exceptions import ClientError

try:
    qs = quicksight_admin.describe_account_settings(AwsAccountId=ACCOUNT_ID)
    edition = qs['AccountSettings'].get('Edition', 'Unknown')
    print(f'\u2713 QuickSight active (Edition: {edition})')
    if edition == 'STANDARD':
        print('  \u26a0 Definition API requires Enterprise edition')
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print('\u2717 QuickSight not subscribed: https://quicksight.aws.amazon.com/')
    else: raise

✓ QuickSight active (Edition: ENTERPRISE)


## 3. Verify Inference Data in Athena

In [15]:
# Build the entire dashboard in one shot.
# gov.create_dashboard() is self-contained -- it creates its own boto3 clients,
# resolves the account, checks QuickSight subscription, verifies Athena data,
# creates all datasets/views/analysis/dashboard, and returns a summary dict.
#
# This is the same entry point that `main.py dashboard create` uses.

result = gov.create_dashboard(region=AWS_DEFAULT_REGION)

print()
print('=' * 70)
print('Dashboard build complete')
print('=' * 70)
print(f"QuickSight subscribed: {result['quicksight_subscribed']} (edition: {result['quicksight_edition']})")
print(f"Dashboard URL:  {result['dashboard_url']}")
print(f"Dashboard ARN:  {result['dashboard_arn']}")
print(f"Analysis ARN:   {result['analysis_arn']}")
print()
print('Datasets:')
for k in ('inference', 'drift', 'feature_drift', 'feature_level', 'accuracy'):
    arn_key = f'{k}_dataset_arn'
    print(f"  {k:14s} {result.get(arn_key, 'N/A')}")
if result.get('embed_url'):
    print(f"\nEmbed URL (valid 10h):\n  {result['embed_url']}")

2026-07-21 21:53:40,046 - src.governance.create_governance_dashboard - INFO - Account:                    329430715989


2026-07-21 21:53:40,047 - src.governance.create_governance_dashboard - INFO - Data-plane region (Athena): us-east-1


2026-07-21 21:53:40,048 - src.governance.create_governance_dashboard - INFO - QuickSight identity region: us-east-1


2026-07-21 21:53:40,094 - src.governance.create_governance_dashboard - INFO - ✓ QuickSight active (Edition: ENTERPRISE)


2026-07-21 21:53:40,095 - src.governance.create_governance_dashboard - INFO - Checking inference_responses table...


...

2026-07-21 21:55:02,932 - src.governance.create_governance_dashboard - INFO -   ✓ Published version 7


2026-07-21 21:55:02,933 - src.governance.create_governance_dashboard - INFO - ✓ Dashboard: https://us-east-1.quicksight.aws.amazon.com/sn/dashboards/fraud-governance-dashboard


2026-07-21 21:55:03,003 - src.governance.create_governance_dashboard - WARNING - Could not generate embed URL: Invalid QuickSight user ARN: null


2026-07-21 21:55:03,003 - src.governance.create_governance_dashboard - WARNING - Ensure embedding is enabled in QuickSight admin settings.



Dashboard build complete
QuickSight subscribed: True (edition: ENTERPRISE)
Dashboard URL:  https://us-east-1.quicksight.aws.amazon.com/sn/dashboards/fraud-governance-dashboard
Dashboard ARN:  arn:aws:quicksight:us-east-1:329430715989:dashboard/fraud-governance-dashboard
Analysis ARN:   arn:aws:quicksight:us-east-1:329430715989:analysis/fraud-governance-analysis

Datasets:
  inference      arn:aws:quicksight:us-east-1:329430715989:dataset/fraud-governance-inference-dataset
  drift          arn:aws:quicksight:us-east-1:329430715989:dataset/fraud-governance-drift-dataset
  feature_drift  arn:aws:quicksight:us-east-1:329430715989:dataset/fraud-governance-feature-drift-dataset
  feature_level  arn:aws:quicksight:us-east-1:329430715989:dataset/fraud-governance-feature-level-dataset
  accuracy       arn:aws:quicksight:us-east-1:329430715989:dataset/fraud-governance-inference-dataset-accuracy


## 9. Publish Dashboard via Definition API

In [31]:
CONFIRM_DELETE = False  # set True to delete all governance QuickSight resources

if CONFIRM_DELETE:
    outcome = gov.delete_dashboard(region=AWS_DEFAULT_REGION)
    print('Deleted:  ', outcome['deleted'])
    print('Not found:', outcome['not_found'])
    print('Errors:   ', outcome['errors'])
else:
    print('Cleanup skipped — set CONFIRM_DELETE = True to run.')

Cleanup skipped — set CONFIRM_DELETE = True to run.
